<a href="https://colab.research.google.com/github/ado1d/suno-ai-test/blob/main/HeartMuLa_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 HeartMuLa-Studio on Colab (Free T4 GPU)

Self-hosted Suno-like AI music generation, running 100% free on Google Colab.

## What's fixed in v2
- ✅ **Reliable Ollama install** — direct binary download with 3 fallbacks (no more silent failures)
- ✅ **Absolute path usage** — `/usr/local/bin/ollama` everywhere (no PATH lookup issues)
- ✅ **Health polling** — waits up to 30s for Ollama to actually respond before continuing
- ✅ **Auto-wipe partial model folders** — fixes `HeartCodec weights not found` permanently
- ✅ **Google Drive persistence** — models + llama3.2 survive disconnects

## Quick start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run **Cell 1** (the big one) — wait ~20 min first time, ~3 min on resume
3. Run **Cell 2** (status check) — repeat until you see `READY`
4. Click the public URL printed at the bottom of Cell 1's output
5. Run **Cell 3** (keepalive) in a separate tab to prevent idle disconnects

## Storage layout on your Drive
- `MyDrive/heartmula_models/`  → ~5 GB (the AI models)
- `MyDrive/heartmula_ollama/`  → ~2 GB (llama3.2 for lyrics)
- `MyDrive/heartmula_songs/`   → your generated songs

Make sure you have **at least 8 GB free** on Drive before running.


In [53]:
# ============================================================
# CELL 1 — ONE-CLICK SETUP (v2: fixed Ollama install)
# First run: ~20 min. Resume: ~3 min (models load from Drive).
# Idempotent — safe to re-run.
# ============================================================
import os, subprocess, time, shutil, sys

def run(cmd, check=False):
    """Run command, print exit code if non-zero, return result."""
    print(f"$ {cmd[:120]}{'...' if len(cmd)>120 else ''}")
    r = subprocess.run(cmd, shell=True, check=check)
    if r.returncode != 0:
        print(f"  ⚠️ exit code {r.returncode}")
    return r

def bg(cmd, log='/tmp/bg.log'):
    """Run command in background."""
    return subprocess.Popen(
        f'nohup {cmd} > {log} 2>&1 &',
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

OLLAMA_BIN = '/usr/local/bin/ollama'

def ollama_works():
    """Verify ollama binary is actually callable."""
    if not os.path.exists(OLLAMA_BIN):
        return False
    r = subprocess.run([OLLAMA_BIN, '--version'], capture_output=True, text=True)
    return r.returncode == 0

# ─── 1. GPU CHECK ─────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "❌ No GPU. Go to: Runtime → Change runtime type → T4 GPU → Save"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)\n")

# ─── 2. MOUNT DRIVE ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODELS = '/content/drive/MyDrive/heartmula_models'
DRIVE_OLLAMA = '/content/drive/MyDrive/heartmula_ollama'
DRIVE_SONGS  = '/content/drive/MyDrive/heartmula_songs'

for d in [DRIVE_MODELS, DRIVE_OLLAMA, DRIVE_SONGS]:
    os.makedirs(d, exist_ok=True)

# Tell Ollama to store its models on Drive
os.environ['OLLAMA_MODELS'] = DRIVE_OLLAMA

df = subprocess.run(['df', '-h', '/content/drive'], capture_output=True, text=True).stdout
print(f"Drive free space:\n{df}")

# ─── 3. INSTALL OLLAMA (properly, with zstd) ────────────────
# Install zstd first (needed to extract ollama's tarball)
subprocess.run('apt-get install -y zstd > /tmp/zstd.log 2>&1', shell=True)
print("✓ zstd installed (needed for ollama extraction)")

if not ollama_works():
    print("\n→ Installing Ollama...")
    # Method 1: Download tar.zst from ollama.com (the CORRECT URL)
    # File is a tarball, not a raw binary. Contains: ./ollama + ./lib/*
    r1 = run(f'curl -L --fail -o /tmp/ollama.tar.zst https://ollama.com/download/ollama-linux-amd64.tar.zst')
    if r1.returncode == 0 and os.path.exists('/tmp/ollama.tar.zst') and os.path.getsize('/tmp/ollama.tar.zst') > 1024*1024:
        # Extract to /tmp/ollama_extract/
        run('rm -rf /tmp/ollama_extract && mkdir -p /tmp/ollama_extract')
        # Modern tar supports --zstd directly
        r_extract = run('tar --zstd -xf /tmp/ollama.tar.zst -C /tmp/ollama_extract')
        if r_extract.returncode != 0:
            # Fallback: pipe through zstd
            run('zstd -d /tmp/ollama.tar.zst -o /tmp/ollama.tar && tar -xf /tmp/ollama.tar -C /tmp/ollama_extract')
        # Find the ollama binary in the extracted tree
        ollama_src = None
        for root, dirs, files in os.walk('/tmp/ollama_extract'):
            if 'ollama' in files:
                ollama_src = os.path.join(root, 'ollama')
                break
        if ollama_src:
            run(f'cp {ollama_src} {OLLAMA_BIN} && chmod +x {OLLAMA_BIN}')
            print(f"  ✓ Extracted from {ollama_src}")
        else:
            print("  ⚠️ Could not find ollama binary in extracted tarball")
    else:
        print("  ⚠️ Direct .tar.zst download failed, trying install script...")

    # Method 2: Fall back to official install script (now zstd is installed)
    if not ollama_works():
        print("  → Trying official install script...")
        r2 = run('curl -fsSL https://ollama.com/install.sh | sh')
        if r2.returncode != 0:
            print("  ⚠️ Install script also failed")

    # Method 3: Last resort — old .tgz URL (works for some older versions)
    if not ollama_works():
        print("  → Trying legacy .tgz URL...")
        run('curl -L --fail -o /tmp/ollama.tgz https://ollama.com/download/ollama-linux-amd64.tgz')
        run('rm -rf /tmp/ollama_extract && mkdir -p /tmp/ollama_extract')
        run('tar -xzf /tmp/ollama.tgz -C /tmp/ollama_extract')
        for root, dirs, files in os.walk('/tmp/ollama_extract'):
            if 'ollama' in files:
                run(f'cp {os.path.join(root, "ollama")} {OLLAMA_BIN} && chmod +x {OLLAMA_BIN}')
                break

    # Final verification
    if ollama_works():
        v = subprocess.run([OLLAMA_BIN, '--version'], capture_output=True, text=True)
        version_str = v.stdout.strip() or v.stderr.strip()
        print(f"\n✓ Ollama installed at {OLLAMA_BIN}")
        print(f"  Version: {version_str}")
    else:
        raise RuntimeError(
            "❌ Could not install Ollama after 3 attempts.\n"
            "Most likely cause: Colab's network can't reach ollama.com.\n"
            "Fix: Runtime → Disconnect and delete runtime → re-run Cell 1."
        )
else:
    v = subprocess.run([OLLAMA_BIN, '--version'], capture_output=True, text=True)
    print(f"✓ Ollama already installed: {v.stdout.strip() or v.stderr.strip()}")

# ─── 4. INSTALL NODE + AUDIO LIBS ────────────────────────────
if not os.path.exists('/usr/bin/node'):
    print("\n→ Installing Node 20...")
    run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs')
else:
    node_v = subprocess.run(['node','-v'], capture_output=True, text=True).stdout.strip()
    print(f"✓ Node already installed: {node_v}")

run('apt-get install -y ffmpeg libsndfile1 > /tmp/apt.log 2>&1')
print("✓ ffmpeg + libsndfile1 ready\n")

# ─── 5. CLONE REPO ────────────────────────────────────────────
REPO = '/content/HeartMuLa-Studio'
if not os.path.exists(REPO):
    print("→ Cloning HeartMuLa-Studio...")
    run(f'git clone https://github.com/fspecii/HeartMuLa-Studio.git {REPO}')
else:
    print(f"✓ Repo already at {REPO}")

# ─── 6. INSTALL PYTHON DEPS ───────────────────────────────────
print("\n→ Installing Python deps (3-10 min first time, ~30s on resume)...")
r = run('pip install -r /content/HeartMuLa-Studio/backend/requirements.txt -q')
if r.returncode != 0:
    print("⚠️ Quiet install had issues, retrying verbose...")
    run('pip install -r /content/HeartMuLa-Studio/backend/requirements.txt')
print("✓ Python deps installed\n")

# ─── 7. BUILD FRONTEND ────────────────────────────────────────
if not os.path.exists(f'{REPO}/frontend/dist'):
    print("→ Building frontend (~1 min)...")
    run('cd /content/HeartMuLa-Studio/frontend && npm install && npm run build')
else:
    print("✓ Frontend already built")

# ─── 8. SYMLINK MODELS TO DRIVE + AUTO-WIPE PARTIALS ─────────
LOCAL_MODELS = f'{REPO}/backend/models'

# If a real (non-symlink) folder exists, move its contents to Drive
if os.path.exists(LOCAL_MODELS) and not os.path.islink(LOCAL_MODELS):
    print("→ Moving existing local models to Drive...")
    for item in os.listdir(LOCAL_MODELS):
        src = os.path.join(LOCAL_MODELS, item)
        dst = os.path.join(DRIVE_MODELS, item)
        if not os.path.exists(dst):
            shutil.move(src, dst)
            print(f"  Moved: {item}")
        else:
            print(f"  {item}: already on Drive, keeping Drive version")
    try:
        os.rmdir(LOCAL_MODELS)
    except OSError:
        shutil.rmtree(LOCAL_MODELS)

# Create symlink: local folder → Drive
if not os.path.islink(LOCAL_MODELS):
    os.symlink(DRIVE_MODELS, LOCAL_MODELS)
print(f"✓ Models symlinked: {LOCAL_MODELS} → {DRIVE_MODELS}")

# 🛡️ AUTO-WIPE PARTIAL MODEL FOLDERS (the HeartCodec fix!)
# Verifies weights files are actually present AND >100 MB.
EXPECTED_FILES = {
    'HeartCodec-oss':                    ['HeartMula_codec.safetensors', 'model.safetensors'],
    'HeartMuLa-oss-3B-happy-new-year':   ['model-00001-of-00004.safetensors'],
    'MuQ':                               ['model.safetensors'],
    'MuLan':                             ['model.safetensors'],
}
print("\n→ Checking model integrity on Drive...")
for folder, required in EXPECTED_FILES.items():
    path = os.path.join(DRIVE_MODELS, folder)
    if not os.path.exists(path):
        print(f"  ⏭️  {folder}: not downloaded yet (will download)")
        continue
    files = os.listdir(path) if os.path.isdir(path) else []
    weights_ok = any(
        os.path.exists(os.path.join(path, f)) and
        os.path.getsize(os.path.join(path, f)) > 100 * 1024 * 1024
        for f in required
    )
    if weights_ok:
        total_size = sum(
            os.path.getsize(os.path.join(path, f))
            for f in files
            if os.path.isfile(os.path.join(path, f))
        )
        print(f"  ✅ {folder}: complete ({total_size/1e9:.2f} GB)")
    else:
        print(f"  ❌ {folder}: incomplete ({files}), WIPING for re-download")
        shutil.rmtree(path)

# ─── 9. WRITE .env ────────────────────────────────────────────
env_content = """LLM_PROVIDER=ollama
OLLAMA_HOST=http://127.0.0.1:11434
OLLAMA_MODEL=llama3.2
HEARTMULA_SEQUENTIAL_OFFLOAD=auto
HEARTMULA_COMPILE=false
"""
os.makedirs(f'{REPO}/backend', exist_ok=True)
with open(f'{REPO}/backend/.env', 'w') as f:
    f.write(env_content)
print("\n✓ .env written")

# ─── 10. START OLLAMA + PULL LLAMA3.2 ─────────────────────────
# Kill any existing ollama process first
subprocess.run('pkill -f "ollama serve" 2>/dev/null', shell=True)
time.sleep(2)

print("\n→ Starting Ollama server...")
bg(f'{OLLAMA_BIN} serve', '/tmp/ollama.log')

# Poll for Ollama to actually be ready (up to 30s)
print("  Waiting for Ollama to respond...")
ollama_ready = False
for i in range(15):
    time.sleep(2)
    check = subprocess.run(
        ['curl', '-s', 'http://127.0.0.1:11434/api/tags'],
        capture_output=True, text=True
    )
    if check.returncode == 0 and check.stdout.strip():
        print(f"✓ Ollama responding (after {2*(i+1)}s)")
        ollama_ready = True
        break

if not ollama_ready:
    print("⚠️ Ollama not responding after 30s. Checking log:")
    run('tail -20 /tmp/ollama.log')

# Check if llama3.2 is already on Drive
llama_list = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
if 'llama3.2' in llama_list.stdout:
    print("✓ llama3.2 already on Drive")
else:
    print("→ Pulling llama3.2 (~2 GB, first time only)...")
    pull_result = run(f'{OLLAMA_BIN} pull llama3.2')
    if pull_result.returncode == 0:
        print("✓ llama3.2 pulled")
    else:
        print("⚠️ Pull failed — you can retry later with: !ollama pull llama3.2")

print("✓ Ollama ready at http://127.0.0.1:11434")

# ─── 11. START BACKEND ────────────────────────────────────────
subprocess.run('pkill -f "uvicorn backend" 2>/dev/null', shell=True)
time.sleep(2)

print("\n→ Starting backend...")
bg('cd /content/HeartMuLa-Studio && python -m uvicorn backend.app.main:app --host 0.0.0.0 --port 8000',
   '/tmp/backend.log')
print("✓ Backend starting (models load from Drive — check STATUS cell for progress)")

# ─── 12. START LOCALTUNNEL ────────────────────────────────────
if subprocess.run('which lt', shell=True, capture_output=True).returncode != 0:
    print("\n→ Installing localtunnel...")
    run('npm install -g localtunnel')

subprocess.run('pkill -f "lt --port" 2>/dev/null', shell=True)
time.sleep(2)

print("→ Starting public URL tunnel...")
bg('lt --port 8000', '/tmp/lt.log')
time.sleep(8)

# ─── 13. SHOW URL ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("🌍 PUBLIC URL — click this to open HeartMuLa-Studio:")
print("=" * 60)
try:
    with open('/tmp/lt.log') as f:
        content = f.read()
    found = False
    for line in content.splitlines():
        if 'url is' in line.lower() or 'loca.lt' in line:
            print(f"\n  {line.strip()}\n")
            found = True
            break
    if not found:
        time.sleep(5)
        with open('/tmp/lt.log') as f:
            print(f.read())
except Exception as e:
    print(f"(could not read tunnel URL yet: {e})")
    print("Run the STATUS cell below to see it.")

print("=" * 60)
print("\n📋 NEXT STEPS:")
print("  1. Run CELL 2 (STATUS CHECK) repeatedly until you see")
print("     '✅ READY — open your localtunnel URL!'")
print("  2. Click the localtunnel URL above (or from STATUS cell)")
print("  3. On the 'Click to continue' localtunnel page, click it")
print("  4. Run CELL 3 (KEEPALIVE) in a separate tab")
print("\n⏱️  First run: backend needs ~15 min to download models to Drive")
print("⏱️  Resume runs: backend starts in ~1 min (models already on Drive)")


✓ GPU: Tesla T4 (15.6 GB VRAM)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive free space:
Filesystem      Size  Used Avail Use% Mounted on
drive            15G  1.9G   14G  13% /content/drive

✓ zstd installed (needed for ollama extraction)
✓ Ollama already installed: ollama version is 0.32.14
✓ Node already installed: v20.20.2
$ apt-get install -y ffmpeg libsndfile1 > /tmp/apt.log 2>&1
✓ ffmpeg + libsndfile1 ready

✓ Repo already at /content/HeartMuLa-Studio

→ Installing Python deps (3-10 min first time, ~30s on resume)...
$ pip install -r /content/HeartMuLa-Studio/backend/requirements.txt -q
✓ Python deps installed

✓ Frontend already built
✓ Models symlinked: /content/HeartMuLa-Studio/backend/models → /content/drive/MyDrive/heartmula_models

→ Checking model integrity on Drive...
  ⏭️  HeartCodec-oss: not downloaded yet (will download)
  ⏭️  HeartMuLa-oss-3B-happy-new-year: not downloaded yet (

## Cell 2 — Status check (re-runnable)

Run this repeatedly until the backend log shows `Ready!` and `Uvicorn running`.


In [65]:
# ============================================================
# CELL 2 — STATUS CHECK (re-run as many times as you want)
# ============================================================
import subprocess, os

print("=" * 60)
print("📄 BACKEND LOG (last 40 lines):")
print("=" * 60)
subprocess.run('tail -n 40 /tmp/backend.log 2>/dev/null || echo "(no log yet)"', shell=True)

print("\n" + "=" * 60)
print("🦙 OLLAMA STATUS:")
print("=" * 60)
subprocess.run('curl -s http://127.0.0.1:11434/api/tags 2>/dev/null || echo "Ollama not responding"', shell=True)
print()

print("=" * 60)
print("🌍 PUBLIC URL:")
print("=" * 60)
try:
    with open('/tmp/lt.log') as f:
        for line in f:
            if 'url is' in line.lower() or 'loca.lt' in line:
                print(f"  {line.strip()}")
                break
        else:
            print("  (no URL yet — tunnel still starting)")
except:
    print("  (no log yet)")

print("\n" + "=" * 60)
print("💾 DRIVE STORAGE:")
print("=" * 60)
subprocess.run('df -h /content/drive', shell=True)

print("\n" + "=" * 60)
print("📦 MODELS ON DRIVE:")
print("=" * 60)
models_dir = '/content/drive/MyDrive/heartmula_models'
if os.path.exists(models_dir):
    subprocess.run(f'ls -la {models_dir}/', shell=True)
    print()
    for d in os.listdir(models_dir):
        full = os.path.join(models_dir, d)
        if os.path.isdir(full):
            size = sum(
                os.path.getsize(os.path.join(full, f))
                for f in os.listdir(full)
                if os.path.isfile(os.path.join(full, f))
            )
            status = "✅" if size > 500*1024*1024 else "❌ incomplete"
            print(f"  {status} {d}: {size/1e9:.2f} GB")
else:
    print("  (Drive not mounted or folder missing)")

print("\n" + "=" * 60)
print("🎯 READY CHECK:")
print("=" * 60)
try:
    with open('/tmp/backend.log') as f:
        log = f.read()
    if 'Uvicorn running' in log and 'Ready!' in log:
        print("  ✅ READY — open your localtunnel URL!")
    elif 'Uvicorn running' in log and 'downloading' in log.lower():
        for line in log.splitlines()[::-1]:
            if 'downloading' in line.lower() or 'Startup' in line:
                print(f"  ⏳ DOWNLOADING: {line.strip()}")
                break
    elif 'ERROR' in log or 'error:' in log:
        print("  ❌ ERROR detected — check backend log above")
    else:
        print("  ⏳ Backend still starting up — wait and re-run this cell")
except:
    print("  (no log yet)")


📄 BACKEND LOG (last 40 lines):

🦙 OLLAMA STATUS:

🌍 PUBLIC URL:
  your url is: https://grumpy-times-leave.loca.lt

💾 DRIVE STORAGE:

📦 MODELS ON DRIVE:


🎯 READY CHECK:
  (no log yet)


## Cell 3 — Keepalive (prevents idle disconnect)

Colab disconnects after ~90 min of idle. Run this cell and **leave it running**
in a separate tab — it pings every 60 seconds so your session stays alive
while you use the HeartMuLa-Studio web UI.

**Tip**: Open this notebook in 2 browser tabs:
- Tab 1: run Cell 3 (keepalive) — leave it alone
- Tab 2: run Cell 1 + Cell 2 — interact normally


In [ ]:
# ============================================================
# CELL 3 — KEEPALIVE (run once, leave running)
# ============================================================
import time
from IPython.display import clear_output

print("🔒 Keepalive started.")
print("Don't close this tab — it pings every 60 seconds.")
print("To stop: press the stop button (■) on this cell.\n")

start = time.time()
tick = 0
while True:
    tick += 1
    elapsed_min = (time.time() - start) / 60
    now = time.strftime('%H:%M:%S')
    print(f"⏰ Tick #{tick} | alive for {elapsed_min:.1f} min | {now}")
    time.sleep(60)
    clear_output(wait=True)


## Troubleshooting

### `FileNotFoundError: 'ollama'` (the bug from v1)
**Fixed in v2.** If it still happens (rare network case), run:
```python
!curl -L -o /usr/local/bin/ollama https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64
!chmod +x /usr/local/bin/ollama
!/usr/local/bin/ollama --version
```
Then re-run Cell 1.

### Backend log shows `FileNotFoundError: HeartCodec weights not found`
**Auto-fixed by Cell 1's integrity check.** If it still happens, run:
```python
!rm -rf /content/drive/MyDrive/heartmula_models/HeartCodec-oss
```
Then re-run Cell 1.

### `Google Drive storage quota has been exceeded`
- Go to https://drive.google.com → empty Trash (left sidebar → Trash → ⋮ → Empty trash)
- Delete `/content/drive/MyDrive/heartmula_models/` entirely (forces full re-download)
- Or use a different Google account with more free space

### Localtunnel URL shows error page
- Click "Click to continue" — that's normal
- Or rerun just the tunnel:
```python
!pkill -f "lt --port"; import time; time.sleep(2)
!nohup lt --port 8000 > /tmp/lt.log 2>&1 & sleep 5 && cat /tmp/lt.log
```

### Backend won't start / port 8000 busy
```python
!pkill -f uvicorn; !pkill -f "lt --port"; !pkill -f "ollama serve"
```
Then re-run Cell 1.

### Generation fails with CUDA OOM
Edit `/content/HeartMuLa-Studio/backend/.env` and change:
```
HEARTMULA_SEQUENTIAL_OFFLOAD=true
```
Then restart backend (re-run Cell 1).

### Ollama not responding
```python
!curl http://127.0.0.1:11434/api/tags
!tail -20 /tmp/ollama.log
```
If empty, the server died. Re-run Cell 1.

### Session disconnected — what now?
Don't panic. Your models + llama3.2 are on Drive, so:
1. Open a fresh Colab session
2. Runtime → T4 GPU
3. Re-run Cell 1 → ~3 min (uses Drive cache)
4. Re-run Cell 2 to verify
5. Re-run Cell 3 to keep alive
